# OPALX vs `tps.py` comparison

This notebook compares the final cumulative TPS order from `../scripts/tps.py` against `opalx-sim/pert-test-uniformsphere.stat`, the `.stat` file whose initial conditions best match the current TPS setup. It uses the same plotted quantities as `opalx-comparison.ipynb`: `rms_x`, `rms_y`, `rms_s`, `rms_px`, `rms_py`, `rms_ps`, `energy`, and `dE`.

The first run executes `tps.py` with Gaussian initial sampling and caches the numerical arrays in `output/tps_py_gaussian_matched_pert_test_uniformsphere_K3_N512_800steps_no_softening_results.npz`. Later runs load that cache unless `force=True` is passed to `load_tps_results()`.

`tps.py` sets the initial longitudinal reference momentum from `INITIAL_KINETIC_ENERGY_GEV = 1e-9`, matching `Edes = 1e-9` in `opalx-sim/pert-opalx.in`. It then shift/scales the Gaussian sampled initial particle cloud to match the first row of `pert-test-uniformsphere.stat` in `mean_x/mean_y/mean_s`, `rms_x/rms_y/rms_s`, and `rms_px/rms_py/rms_ps`.

The default plot uses Gaussian initial sampling and finite-size TPS smoothing (`R_SOFT = 2e-4 m`). The smoothing is needed because Gaussian close-particle pairs make the unsmoothed K=2 and K=3 Taylor terms blow up. It compares the natural cumulative orders K=1, K=2, and K=3 with `FIELD_SCALE = 1.0`. The current TPS run caches K=3, 512 particles, and 801 stored points so the 800 integration intervals match the OPALX time-step grid.

In [2]:
from pathlib import Path
import os
import re
import runpy
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


M_E_MEV = 0.511


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "python" / "lw_perturbation").exists() and (path / "python" / "scripts" / "tps.py").exists():
            return path
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
LW_DIR = REPO_ROOT / "python" / "lw_perturbation"
TPS_SCRIPT = REPO_ROOT / "python" / "scripts" / "tps.py"
OPALX_STAT = LW_DIR / "opalx-sim" / "pert-test-uniformsphere.stat"
TPS_INITIAL_DISTRIBUTION = "gaussian"
TPS_PERTURB_ORDER = 3
TPS_R_SOFT = 2.0e-4
TPS_CACHE = LW_DIR / "output" / "tps_py_gaussian_matched_pert_test_uniformsphere_K3_N512_800steps_rsoft2e-4_results.npz"
FIELD_SCALE = 1.0
DIAGNOSTIC_FIELD_SCALE = 1.19
PLOT_PATH = LW_DIR / "comparison-tps-gaussian-pert-test-uniformsphere-K1-K3-N512-800steps-rsoft2e-4.png"
DIAGNOSTIC_PLOT_PATH = LW_DIR / "comparison-tps-gaussian-pert-test-uniformsphere-K1-K3-N512-800steps-rsoft2e-4-field-scale-1p19.png"

Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def read_sdds_file(filename):
    """Read an OPAL/OPALX SDDS-format .stat/.dat file into a DataFrame."""
    col_names = []
    col_info = {}
    header_lines = 0
    first_data_line = None

    with open(filename, "r") as f:
        for line in f:
            header_lines += 1
            stripped = line.strip()

            if stripped.startswith("&column"):
                col_def = stripped
                while "&end" not in col_def:
                    next_line = next(f).strip()
                    header_lines += 1
                    col_def += " " + next_line

                name_match = re.search(r"name\s*=\s*(\S+)", col_def)
                type_match = re.search(r"type\s*=\s*(\S+)", col_def)
                unit_match = re.search(r"units\s*=\s*(\S+)", col_def)
                desc_match = re.search(r'description\s*=\s*"([^"]*)"', col_def)

                name = name_match.group(1).strip(",\"") if name_match else f"col_{len(col_names)}"
                dtype = type_match.group(1).strip(",\"") if type_match else "double"
                unit = unit_match.group(1).strip(",\"") if unit_match else ""
                desc = desc_match.group(1) if desc_match else ""

                col_names.append(name)
                col_info[name] = (dtype, unit, desc)

            if stripped.startswith("&data"):
                while True:
                    next_line = next(f).strip()
                    header_lines += 1
                    parts = next_line.split()
                    if len(parts) >= 2:
                        try:
                            float(parts[0])
                            float(parts[1])
                            first_data_line = next_line
                            break
                        except ValueError:
                            pass
                    if len(parts) == 1:
                        try:
                            float(parts[0])
                            if "." not in parts[0] and "e" not in parts[0].lower():
                                continue
                            first_data_line = next_line
                            break
                        except ValueError:
                            continue
                break

    if first_data_line is None:
        raise RuntimeError(f"Could not find numeric data in SDDS file: {filename}")

    data_rows = [[float(x) for x in first_data_line.split()]]
    with open(filename, "r") as f:
        for _ in range(header_lines):
            next(f)
        for line in f:
            stripped = line.strip()
            if stripped and not stripped.startswith("&"):
                try:
                    values = [float(x) for x in stripped.split()]
                    if len(values) == len(col_names):
                        data_rows.append(values)
                except ValueError:
                    continue

    return pd.DataFrame(data_rows, columns=col_names), col_info


def stat_columns(path=OPALX_STAT):
    _, col_info = read_sdds_file(path)
    return pd.DataFrame(
        {"column": name, "unit": unit, "description": desc}
        for name, (_, unit, desc) in col_info.items()
    )

In [ ]:
def load_tps_results(cache_path=TPS_CACHE, force=False):
    """Load cached `tps.py` arrays or execute `tps.py` and cache its results."""
    cache_path = Path(cache_path)
    if cache_path.exists() and not force:
        with np.load(cache_path, allow_pickle=False) as data:
            return {key: data[key] for key in data.files}

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    old_cwd = Path.cwd()
    old_env = {
        name: os.environ.get(name)
        for name in ("TPS_INITIAL_DISTRIBUTION", "TPS_PERTURB_ORDER", "TPS_R_SOFT", "TPS_USE_CIC_SOFTENING")
    }
    try:
        os.environ["TPS_INITIAL_DISTRIBUTION"] = TPS_INITIAL_DISTRIBUTION
        os.environ["TPS_PERTURB_ORDER"] = str(TPS_PERTURB_ORDER)
        os.environ["TPS_R_SOFT"] = str(TPS_R_SOFT)
        os.environ["TPS_USE_CIC_SOFTENING"] = "1"
        os.chdir(TPS_SCRIPT.parent)
        namespace = runpy.run_path(str(TPS_SCRIPT), run_name="__main__")
    finally:
        os.chdir(old_cwd)
        for name, value in old_env.items():
            if value is None:
                os.environ.pop(name, None)
            else:
                os.environ[name] = value

    required = ["times", "mean_r_orders", "rms_r_orders", "mean_p_orders", "rms_p_orders", "cum_r", "cum_p", "p_t_particles", "K"]
    missing = [name for name in required if name not in namespace]
    if missing:
        raise RuntimeError(f"`tps.py` did not expose expected result arrays: {missing}")

    results = {name: np.asarray(namespace[name]) for name in required}
    np.savez_compressed(cache_path, **results)
    return results


def kinetic_energy_mev(p_mev_c):
    """Relativistic kinetic energy from momentum vectors in MeV/c."""
    p2 = np.sum(np.asarray(p_mev_c) ** 2, axis=-1)
    return np.sqrt(M_E_MEV ** 2 + p2) - M_E_MEV


def cumulative_with_scale(cumulative, order, field_scale=1.0):
    """Recombine cumulative TPS arrays as sum_n field_scale**n * coeff_n."""
    cumulative = np.asarray(cumulative)
    out = cumulative[:, :, 0, :].copy()
    for n in range(1, order + 1):
        coeff_n = cumulative[:, :, n, :] - cumulative[:, :, n - 1, :]
        out += (field_scale ** n) * coeff_n
    return out


def tps_results_to_moments(results, order=None, field_scale=1.0):
    """Convert `tps.py` arrays to the same columns used by opalx-comparison.ipynb."""
    k = int(results["K"]) if order is None else int(order)
    times = np.asarray(results["times"])
    if abs(field_scale - 1.0) < 1e-15:
        rms_r = np.asarray(results["rms_r_orders"])[:, k, :]
        rms_p_bg = np.asarray(results["rms_p_orders"])[:, k, :] / M_E_MEV
    else:
        if "cum_r" not in results or "cum_p" not in results:
            raise ValueError("Scaled TPS moments require cum_r and cum_p; rerun with force_tps=True")
        r_particles = cumulative_with_scale(results["cum_r"], k, field_scale=field_scale)
        p_particles_for_rms = cumulative_with_scale(results["cum_p"], k, field_scale=field_scale)
        rms_r = r_particles.std(axis=1)
        rms_p_bg = p_particles_for_rms.std(axis=1) / M_E_MEV

    if "cum_p" in results:
        p_particles = cumulative_with_scale(results["cum_p"], k, field_scale=field_scale)
    elif k == int(results["K"]):
        p_particles = np.asarray(results["p_t_particles"]).transpose(1, 0, 2)
    else:
        raise ValueError("Cached TPS results do not contain cum_p; rerun with force_tps=True")
    particle_energy = kinetic_energy_mev(p_particles)

    return pd.DataFrame({
        "t": times * 1.0e9,
        "rms_x": rms_r[:, 0],
        "rms_y": rms_r[:, 1],
        "rms_s": rms_r[:, 2],
        "rms_px": rms_p_bg[:, 0],
        "rms_py": rms_p_bg[:, 1],
        "rms_ps": rms_p_bg[:, 2],
        "energy": particle_energy.mean(axis=1),
        "dE": particle_energy.std(axis=1),
    })

In [ ]:
def compare_opalx_to_tps(
    stat_path=OPALX_STAT,
    cache_path=TPS_CACHE,
    columns=None,
    order=(1, 2, 3),
    data_range=None,
    field_scale=1.0,
    save_as=PLOT_PATH,
    ncols=2,
    force_tps=False,
):
    """Compare OPALX .stat output against one or more TPS cumulative orders."""
    if columns is None:
        columns = [
            "rms_x",
            "rms_y",
            "rms_s",
            "rms_px",
            "rms_py",
            "rms_ps",
            "energy",
            "dE",
        ]
    columns = list(columns)

    opalx, _ = read_sdds_file(stat_path)
    tps_results = load_tps_results(cache_path=cache_path, force=force_tps)
    max_order = int(tps_results["K"])
    if order is None:
        orders = [max_order]
    elif np.isscalar(order):
        orders = [int(order)]
    else:
        orders = [int(k) for k in order]
    invalid_orders = [k for k in orders if k < 0 or k > max_order]
    if invalid_orders:
        raise ValueError(f"Requested TPS orders outside cached range 0..{max_order}: {invalid_orders}")
    tps_by_order = {k: tps_results_to_moments(tps_results, order=k, field_scale=field_scale) for k in orders}

    missing_opalx = [col for col in columns if col not in opalx.columns]
    missing_tps = [col for col in columns if any(col not in tps.columns for tps in tps_by_order.values())]
    if missing_opalx:
        raise ValueError(f"Missing OPALX columns: {missing_opalx}")
    if missing_tps:
        raise ValueError(f"Missing TPS columns: {missing_tps}")

    if data_range is None:
        row_slice = slice(None)
    else:
        start, stop = data_range
        row_slice = slice(start, stop)

    opalx_view = opalx.iloc[row_slice].copy()
    opalx_t = opalx_view["t"].to_numpy()
    tps_t = next(iter(tps_by_order.values()))["t"].to_numpy()
    valid_time = (opalx_t >= tps_t.min()) & (opalx_t <= tps_t.max())
    if not np.any(valid_time):
        raise ValueError("OPALX and TPS time grids do not overlap")
    opalx_view = opalx_view.loc[valid_time].copy()
    opalx_t = opalx_view["t"].to_numpy()

    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 150,
        "savefig.dpi": 200,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "grid.linestyle": "--",
    })

    labels = {
        "rms_x": "RMS x [m]",
        "rms_y": "RMS y [m]",
        "rms_s": "RMS z/s [m]",
        "rms_px": "RMS px [beta*gamma]",
        "rms_py": "RMS py [beta*gamma]",
        "rms_ps": "RMS pz/ps [beta*gamma]",
        "energy": "mean kinetic energy [MeV]",
        "dE": "energy spread [MeV]",
    }

    ncols = max(1, min(int(ncols), len(columns)))
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(7 * ncols, 2.6 * nrows),
        sharex=True,
        constrained_layout=True,
        squeeze=False,
    )

    summary_rows = []
    for ax, col in zip(axes.ravel(), columns):
        opalx_y = opalx_view[col].to_numpy()
        ax.plot(opalx_t, opalx_y, label="OPALX .stat", lw=1.8, color="black")
        err_for_axis = []

        for k, tps in tps_by_order.items():
            tps_y = np.interp(opalx_t, tps_t, tps[col].to_numpy())
            abs_err = np.abs(opalx_y - tps_y)
            err_for_axis.append(abs_err)
            scale_label = f", λ={field_scale:g}" if abs(field_scale - 1.0) >= 1e-15 else ""
            ax.plot(opalx_t, tps_y, label=f"TPS K={k}{scale_label}", lw=1.8)

            denom = np.maximum(np.abs(tps_y), 1e-30)
            summary_rows.append({
                "order": k,
                "field_scale": field_scale,
                "column": col,
                "max_abs_error": np.max(abs_err),
                "mean_abs_error": np.mean(abs_err),
                "max_relative_error": np.max(abs_err / denom),
                "mean_relative_error": np.mean(abs_err / denom),
            })

        ax.set_title(col)
        ax.set_ylabel(labels.get(col, col))
        ax.legend(loc="best", framealpha=0.7)

        panel_err = np.min(np.vstack(err_for_axis), axis=0)
        ax_err = ax.twinx()
        ax_err.plot(opalx_t, panel_err, color="silver", lw=1.2, ls="--", alpha=0.8, label="min |Delta|")
        ax_err.set_ylabel("min |Delta|", color="silver", fontsize=9)
        ax_err.tick_params(axis="y", colors="silver", labelsize=8)
        ax_err.yaxis.get_offset_text().set_color("silver")
        if np.any(panel_err > 0):
            ax_err.set_yscale("log")

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)
    for ax in axes[-1, :]:
        if ax.get_visible():
            ax.set_xlabel("t [ns]")

    if save_as is not None:
        save_as = Path(save_as)
        save_as.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_as)
        print(f"saved {save_as}")
    plt.show()

    return pd.DataFrame(summary_rows)

In [ ]:
stat_columns(OPALX_STAT)

In [ ]:
comparison_summary = compare_opalx_to_tps(
    stat_path=OPALX_STAT,
    cache_path=TPS_CACHE,
    field_scale=FIELD_SCALE,
    save_as=PLOT_PATH,
    # force_tps=True,  # uncomment to rerun ../scripts/tps.py instead of loading the cache
)
comparison_summary

# Optional diagnostic from the earlier K=2 scaling check:
# compare_opalx_to_tps(
#     stat_path=OPALX_STAT,
#     cache_path=TPS_CACHE,
#     field_scale=DIAGNOSTIC_FIELD_SCALE,
#     save_as=DIAGNOSTIC_PLOT_PATH,
# )